# Generate the result in RQ2

This script generates the result in RQ2. The script reads the data from the `data/rq2-dp` folder and generates the results presented in the paper. 
- `data/rq2-node-reduce/fuzzer_stats.csv` contains the statistics of the five repetitions of the fuzzing runs with our without applying the node reduction machanism; it is the parsed result of the `fuzzer_stats` file generated by `ALF++`.
- `data/rq2-node-reduce/compile-log/` directory contains the logs of docker image building, which contains the information of the time spent on compiling the subject programs for fuzzing.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Directory where the coverage records are stored
directory = "data/rq2-node-reduce"

fuzz_stats_path = os.path.join(directory, "fuzzer_stats.csv")
df_stats = pd.read_csv(fuzz_stats_path)
df_stats

,subject,node_select,index,total_edges,execs_per_sec,execs_done,container_name,filename,start_time,last_update,...,var_byte_count,havoc_expansion,auto_dict_entries,testcache_size,testcache_count,testcache_evict,afl_banner,afl_version,target_mode,command_line
0,freetype2,no,0,46051,49.08,8502463,gallant_cartwright,gallant_cartwright_fuzzer_stats,1738244961,1738418212,...,3,5,0,2896,2,0,/out/ftfuzzer,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
1,freetype2,no,1,46051,52.86,9160011,pedantic_margulis,pedantic_margulis_fuzzer_stats,1738245317,1738418601,...,3,5,0,2896,2,0,/out/ftfuzzer,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
2,freetype2,no,2,46051,48.04,8323007,nostalgic_ardinghelli,nostalgic_ardinghelli_fuzzer_stats,1738245318,1738418561,...,3,5,0,2896,2,0,/out/ftfuzzer,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
3,freetype2,no,3,46051,52.48,9092718,hardcore_keldysh,hardcore_keldysh_fuzzer_stats,1738245319,1738418595,...,3,5,0,2896,2,0,/out/ftfuzzer,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
4,freetype2,no,4,46051,52.84,9156658,inspiring_jackson,inspiring_jackson_fuzzer_stats,1738245321,1738418603,...,3,5,0,2896,2,0,/out/ftfuzzer,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,libpcap,yes,0,7815,5666.23,1037868678,sweet_engelbart,sweet_engelbart_fuzzer_stats,1740221649,1740404817,...,0,5,0,2,1,0,/out/fuzz_both,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
76,libpcap,yes,1,7815,5664.94,1037689802,xenodochial_euclid,xenodochial_euclid_fuzzer_stats,1740221716,1740404893,...,0,5,0,2,1,0,/out/fuzz_both,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
77,libpcap,yes,2,7815,5657.47,1036566425,romantic_mirzakhani,romantic_mirzakhani_fuzzer_stats,1740221718,1740404939,...,0,5,0,2,1,0,/out/fuzz_both,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...
78,libpcap,yes,3,7815,5671.52,1038943152,gifted_wilson,gifted_wilson_fuzzer_stats,1740221720,1740404906,...,0,5,0,2,1,0,/out/fuzz_both,++4.31a,persistent shmem_testcase deferred,./afl-fuzz -i /out/seeds -o /out/corpus -m non...


## 1. Check the number of nodes with and without the node removal (The first part of Table 3)

In [2]:
unique_check = (
    df_stats.groupby(["subject", "node_select"])["total_edges"]
    .nunique()
    .reset_index(name="unique_count")
)

# Check unique
unique_pairs = unique_check[unique_check["unique_count"] == 1]
assert np.all(unique_check["unique_count"] == 1)

summary_df = (
    df_stats.groupby(["subject", "node_select"])["total_edges"]
    .mean()
    .reset_index()
    .pivot(index="subject", columns="node_select", values="total_edges")
)
summary_df["ratio"] = summary_df["yes"] / summary_df["no"]
display(summary_df)

print("Average ratio:", summary_df["ratio"].mean())

node_select,no,yes,ratio
subject,,,
freetype2,46051.0,26553.0,0.576600
jsoncpp,8780.0,5631.0,0.641344
libjpeg,36840.0,21788.0,0.591422
libpcap,14355.0,7815.0,0.544410
libpng,10463.0,6077.0,0.580809
libxml2,93858.0,50573.0,0.538825
sqlite3,58253.0,32849.0,0.563902
zlib,1775.0,986.0,0.555493


Average ratio: 0.5741005271921845


## 2. Check the time spent on compiling the subject programs (The second part of Table 3)

In [3]:
# parse_log_paths
def parse_log_paths(log_paths):
    # parse filename
    fname = os.path.basename(log_paths)
    fuzzer = fname.split("-")[1]
    idx = int(fname[:-4].split("-")[-1])
    if "jsoncpp" in fname:
        subject = "jsoncpp"
    elif "freetype2" in fname:
        subject = "freetype2"
    elif "libjpeg" in fname:
        subject = "libjpeg"
    elif "libpcap" in fname:
        subject = "libpcap"
    elif "libpng" in fname:
        subject = "libpng"
    elif "libxml2" in fname:
        subject = "libxml2"
    elif "zlib" in fname:
        subject = "zlib"
    else:
        subject = "sqlite3"

    with open(log_paths, "r") as f:
        lines = f.readlines()
    # check the line with #22 DONE <- the docker build instruction that
    # compiles the subject
    done_lines = [line for line in lines if "#22 DONE" in line]
    assert len(done_lines) == 1
    done_line = done_lines[0]
    # parse compile time
    ctime = float(done_line.strip().split()[-1][:-1])
    return {
        "fuzzer": (
            "no_selection"
            if fuzzer == "aflmmbb_no_selection"
            else "with_selection"
        ),
        "idx": idx,
        "subject": subject,
        "compile_time": ctime,
    }


log_path_temp = os.path.join(directory, "compile-log", "*.log")
log_paths = glob.glob(log_path_temp)
data = []
for log_path in log_paths:
    data.append(parse_log_paths(log_path))
df = pd.DataFrame(data)

summary_df = (
    df.groupby(["fuzzer", "subject"])
    .mean()
    .pivot_table(index="subject", columns="fuzzer", values="compile_time")
)
# order
summary_df = summary_df.reindex(
    [
        "sqlite3",
        "freetype2",
        "libxml2",
        "libjpeg",
        "libpcap",
        "zlib",
        "libpng",
        "jsoncpp",
    ]
)
summary_df["ratio"] = (
    summary_df["with_selection"] / summary_df["no_selection"] - 1
) * 100
summary_df["delta"] = summary_df["with_selection"] - summary_df["no_selection"]
summary_df.loc["average"] = summary_df.mean()
display(summary_df)

fuzzer,no_selection,with_selection,ratio,delta
subject,,,,
sqlite3,286.2800,203.6400,-28.866844,-82.640
freetype2,83.7800,76.1600,-9.095249,-7.620
libxml2,50.5200,44.8000,-11.322249,-5.720
libjpeg,53.6800,43.8000,-18.405365,-9.880
libpcap,58.2000,49.2600,-15.360825,-8.940
zlib,4.7600,4.1400,-13.025210,-0.620
libpng,28.5600,25.8200,-9.593838,-2.740
jsoncpp,40.0400,36.5600,-8.691309,-3.480
average,75.7275,60.5225,-14.295111,-15.205


## 3. Check the exec_per_sec of the fuzzing runs with and without the node removal (The third part of Table 3)

In [4]:
# summary
summary_df = (
    df_stats.groupby(["subject", "node_select"])["execs_per_sec"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
    .pivot(
        index="subject",
        columns="node_select",
        values=["mean", "std", "min", "max"],
    )
)
# ratio of mean execs_per_sec
summary_df["ratio"] = summary_df["mean"]["yes"] / summary_df["mean"]["no"]
display(summary_df)
print("Average ratio:", summary_df["ratio"].mean())

mean                   std                   min           \
node_select        no       yes          no          yes       no      yes   
subject                                                                      
freetype2      51.060   104.048    2.316549     2.475029    48.04   100.92   
jsoncpp      2686.258  3370.076   85.912692   109.898828  2601.59  3189.00   
libjpeg       290.404  1061.582   32.488489   124.988699   238.22   855.21   
libpcap      5415.946  5664.236   12.621225     5.330331  5396.15  5657.47   
libpng       2210.390  3140.782  106.187380   152.986180  2099.94  2968.03   
libxml2        18.818    50.170    0.711456     2.256690    18.07    47.34   
sqlite3        21.932    31.430    0.472832     0.472335    21.16    31.00   
zlib         7043.662  7683.914  639.002202  1919.852399  6701.35  6824.41   

                 max               ratio  
node_select       no       yes            
subject                                   
freetype2      52.86    107.66  2.037759  
jsoncpp      2826.35   3462.96  1.254562  
libjpeg       317.82   1164.75  3.655535  
libpcap      5430.54   5671.52  1.045844  
libpng       2349.76   3342.17  1.420918  
libxml2        19.80     53.23  2.666064  
sqlite3        22.38     32.20  1.433066  
zlib         8184.83  11118.25  1.090898

Average ratio: 1.8255807219464426
